In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import pybedtools
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

# Было принято решение не использовать отдельную аннотацию HGNC. Будем брать официальный gene_name из hencode 47, так как он так или иначе синхронизирован с HGNC. \
# Уменьшение числа gene_name - это итог хакономерной политики gencode.

In [2]:
ROOT = Path().cwd()
DATA = Path("../data").resolve()
DATA_INITIAL = Path("../data_initial").resolve()
FANTOM = DATA / "fantom_cat"
FANTOM_INITIAL = DATA_INITIAL / "fantom_cat"
ANNO = Path("/export/home/public/agletdinov_shared/annotations")
ENSEMBL = DATA / "ensembl"
ENSEMBL_INITIAL = DATA_INITIAL / "ensembl"
GENCODE = DATA / "gencode"
GENCODE_INITIAL = DATA_INITIAL / "gencode"
ID_HISTORY_CONVERTER_RES = DATA / "id_history_converter_res"
HGNC_INITIAL = DATA_INITIAL / "HGNC"

In [3]:
for directory in DATA_INITIAL, FANTOM_INITIAL, DATA, FANTOM, ENSEMBL_INITIAL, ENSEMBL, GENCODE_INITIAL, GENCODE, ID_HISTORY_CONVERTER_RES, HGNC_INITIAL:
    directory.mkdir(exist_ok=True, parents=True)

In [54]:
def gencode_gtf_processing(to_gtf_file):
    gtf_file_GENCODE = to_gtf_file
    anno_GENCODE = pd.read_csv(gtf_file_GENCODE, sep="\t", header=None, skiprows=5)
    anno_GENCODE["ens_gene"] = anno_GENCODE[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])
    anno_GENCODE["gene_name"] = anno_GENCODE[8].apply(lambda x: x.split("gene_name")[1].split(" ")[1].replace('"', '').replace(';', ''))
    anno_GENCODE["unique_genome_reg_name"] = anno_GENCODE["ens_gene"] + "-" + anno_GENCODE[2]
    gencode_hg38 = anno_GENCODE[[0, 3, 4, "unique_genome_reg_name", 5, 6, "ens_gene", "gene_name"]]
    mask = gencode_hg38["unique_genome_reg_name"].apply(lambda x: "gene" in x)
    gencode_genes_hg38 = gencode_hg38[mask]
    gencode_genes_hg38.head(2)
    gencode_genes_hg38 = gencode_genes_hg38.rename(columns={0:"chrom", 3:"start", 4:"end", "unique_genome_reg_name":"name", 5:"score", 6:"strand"})
    gencode_genes_hg38 = gencode_genes_hg38.drop_duplicates()
    gencode_genes_hg38 = gencode_genes_hg38.drop_duplicates()
    gencode_genes_hg38["gene_ens--gene_name"] = gencode_genes_hg38["ens_gene"] + "--" + gencode_genes_hg38["gene_name"]
    return gencode_genes_hg38

In [60]:
!wget https://storage.googleapis.com/public-download-files/hgnc/archive/archive/monthly/tsv/hgnc_complete_set_2024-10-01.txt -O {HGNC_INITIAL}/hgnc_complete_set_2024-10-01.txt

--2025-12-06 15:38:34--  https://storage.googleapis.com/public-download-files/hgnc/archive/archive/monthly/tsv/hgnc_complete_set_2024-10-01.txt
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.205.207, 64.233.161.207, 173.194.222.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.205.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16204004 (15M) ["text/plain]
Saving to: ‘/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/HGNC/hgnc_complete_set_2024-10-01.txt’

100%[======================================>] 16,204,004  26.5MB/s   in 0.6s   

2025-12-06 15:38:35 (26.5 MB/s) - ‘/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/HGNC/hgnc_complete_set_2024-10-01.txt’ saved [16204004/16204004]



In [61]:
to_file = HGNC_INITIAL / "hgnc_complete_set_2024-10-01.txt"
hgnc_complete_set = pd.read_csv(to_file, sep="\t")
hgnc_complete_set.shape

/tmp/ipykernel_114947/1302573397.py:2: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc_complete_set = pd.read_csv(to_file, sep="\t")


(43882, 54)

In [62]:
hgnc_complete_set.head(3)

,hgnc_id,symbol,name,locus_group,locus_type,status,location,location_sortable,alias_symbol,alias_name,...,cd,lncrnadb,enzyme_id,intermediate_filament_db,rna_central_id,lncipedia,gtrnadb,agr,mane_select,gencc
0,HGNC:5,A1BG,alpha-1-B glycoprotein,protein-coding gene,gene with protein product,Approved,19q13.43,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HGNC:5,ENST00000263100.8|NM_130786.4,NaN
1,HGNC:37133,A1BG-AS1,A1BG antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,19q13.43,NaN,FLJ23569,NaN,...,NaN,NaN,NaN,NaN,URS00007E4F6E,A1BG-AS1,NaN,HGNC:37133,NaN,NaN
2,HGNC:24086,A1CF,APOBEC1 complementation factor,protein-coding gene,gene with protein product,Approved,10q11.23,NaN,ACF|ASP|ACF64|ACF65|APOBEC1CF,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HGNC:24086,ENST00000373997.8|NM_014576.4,NaN


In [63]:
columns = ["symbol", "name", "alias_symbol", "prev_symbol", "ensembl_gene_id"]
hgnc_complete_set = hgnc_complete_set[columns]

In [64]:
hgnc_complete_set_with_ensg = hgnc_complete_set.dropna(subset=["ensembl_gene_id"])
hgnc_complete_set_with_ensg.shape

(41244, 5)

In [65]:
gtf_file_GENCODE = GENCODE_INITIAL / "gencode.v47.basic.annotation.gtf.gz"
gencode_genes_hg38 = gencode_gtf_processing(gtf_file_GENCODE)
gencode_genes_hg38

,chrom,start,end,name,score,strand,ens_gene,gene_name,gene_ens--gene_name
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825,DDX11L16,ENSG00000290825--DDX11L16
6,chr1,12010,13670,ENSG00000223972-gene,.,+,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1
14,chr1,14356,30744,ENSG00000310526-gene,.,-,ENSG00000310526,WASH7P,ENSG00000310526--WASH7P
108,chr1,14696,24886,ENSG00000227232-gene,.,-,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P
120,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1
...,...,...,...,...,...,...,...,...,...
2225559,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6
2225564,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE
2225567,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB
2225572,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT


In [89]:
anno_GENCODE[anno_GENCODE["ens_gene"] == "ENSG00000217801"].loc[1782, 8]

'gene_id "ENSG00000217801.13"; gene_type "transcribed_unprocessed_pseudogene"; gene_name "ENSG00000217801"; level 2; havana_gene "OTTHUMG00000000629.5";'

In [67]:
mask = hgnc_complete_set_with_ensg["ensembl_gene_id"].isin(gencode_genes_hg38["ens_gene"])
hgnc_complete_set_with_ensg[~mask]

,symbol,name,alias_symbol,prev_symbol,ensembl_gene_id
914,ALOX12-AS1,ALOX12 antisense RNA 1,NaN,NaN,ENSG00000215067
972,FAM41AY2,"family with sequence similarity 41 member A, Y...",NaN,NaN,ENSG00000226362
1150,FAM153B,family with sequence similarity 153 member B,NaN,NaN,ENSG00000182230
1216,FAM197Y10,family with sequence similarity 197 Y-linked m...,NaN,NaN,ENSG00000235719
1289,FAM239A,family with sequence similarity 239 member A,NaN,NaN,ENSG00000234449
...,...,...,...,...,...
41634,DDIT4-AS1,DDIT4 antisense RNA 1,RP11-442H21.2,NaN,ENSG00000269926
41668,DDX11L3,DEAD/H-box helicase 11 like 3 (pseudogene),NaN,NaN,ENSG05220017861
41832,DENND6A-AS1,DENND6A antisense RNA 1,NaN,NaN,ENSG00000239801
43330,ENTPD3-AS1,"ENTPD3, EIF1B and MYRIP antisense RNA 1",FLJ36665,EIF1B-AS1,ENSG00000223797


In [69]:
gencode_genes_hg38_merged_with_hgnc = pd.merge(left=gencode_genes_hg38, right=hgnc_complete_set_with_ensg[["ensembl_gene_id","symbol","alias_symbol","prev_symbol"]], how="left", left_on="ens_gene", right_on="ensembl_gene_id")
gencode_genes_hg38_merged_with_hgnc

,chrom,start,end,name,score,strand,ens_gene,gene_name,gene_ens--gene_name,ensembl_gene_id,symbol,alias_symbol,prev_symbol
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825,DDX11L16,ENSG00000290825--DDX11L16,NaN,NaN,NaN,NaN
1,chr1,12010,13670,ENSG00000223972-gene,.,+,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1,ENSG00000223972,DDX11L1,NaN,NaN
2,chr1,14356,30744,ENSG00000310526-gene,.,-,ENSG00000310526,WASH7P,ENSG00000310526--WASH7P,NaN,NaN,NaN,NaN
3,chr1,14696,24886,ENSG00000227232-gene,.,-,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P,ENSG00000227232,WASH7P,FAM39F,NaN
4,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1,ENSG00000278267,MIR6859-1,hsa-mir-6859-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
78720,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6,ENSG00000198695,MT-ND6,NAD6|ND6,MTND6
78721,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE,ENSG00000210194,MT-TE,trnE,MTTE
78722,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB,ENSG00000198727,MT-CYB,COB|CYTB|UQCR3,MTCYB
78723,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT,ENSG00000210195,MT-TT,trnT,MTTT|LIMM


In [81]:
gencode_genes_hg38_merged_with_hgnc["ens_gene"].nunique()

78724

In [82]:
gencode_genes_hg38_merged_with_hgnc["ensembl_gene_id"].nunique()

41056

In [76]:
mask = (gencode_genes_hg38_merged_with_hgnc["gene_name"].apply(lambda x: "ENSG" not in x))
gencode_genes_hg38_merged_with_hgnc[mask]

,chrom,start,end,name,score,strand,ens_gene,gene_name,gene_ens--gene_name,ensembl_gene_id,symbol,alias_symbol,prev_symbol
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825,DDX11L16,ENSG00000290825--DDX11L16,NaN,NaN,NaN,NaN
1,chr1,12010,13670,ENSG00000223972-gene,.,+,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1,ENSG00000223972,DDX11L1,NaN,NaN
2,chr1,14356,30744,ENSG00000310526-gene,.,-,ENSG00000310526,WASH7P,ENSG00000310526--WASH7P,NaN,NaN,NaN,NaN
3,chr1,14696,24886,ENSG00000227232-gene,.,-,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P,ENSG00000227232,WASH7P,FAM39F,NaN
4,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1,ENSG00000278267,MIR6859-1,hsa-mir-6859-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
78720,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6,ENSG00000198695,MT-ND6,NAD6|ND6,MTND6
78721,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE,ENSG00000210194,MT-TE,trnE,MTTE
78722,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB,ENSG00000198727,MT-CYB,COB|CYTB|UQCR3,MTCYB
78723,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT,ENSG00000210195,MT-TT,trnT,MTTT|LIMM


In [83]:
mask = (gencode_genes_hg38_merged_with_hgnc["gene_name"] != gencode_genes_hg38_merged_with_hgnc["symbol"])
gencode_genes_hg38_merged_with_hgnc[mask].dropna(subset="ensembl_gene_id")

,chrom,start,end,name,score,strand,ens_gene,gene_name,gene_ens--gene_name,ensembl_gene_id,symbol,alias_symbol,prev_symbol
93,chr1,1063079,1069355,ENSG00000217801-gene,.,+,ENSG00000217801,ENSG00000217801,ENSG00000217801--ENSG00000217801,ENSG00000217801,PTPN11BP,NaN,NaN
1021,chr1,24496254,24556046,ENSG00000286061-gene,.,-,ENSG00000286061,NCMAP-DT,ENSG00000286061--NCMAP-DT,ENSG00000286061,RCAN3AS,TCONS_00001428,NCMAP-DT
2408,chr1,65279456,65306521,ENSG00000229294-gene,.,-,ENSG00000229294,ENSG00000229294,ENSG00000229294--ENSG00000229294,ENSG00000229294,DNAJC6-AS1,AL139294.1,NaN
6559,chr1,231223763,231241187,ENSG00000143633-gene,.,-,ENSG00000143633,C1orf131,ENSG00000143633--C1orf131,ENSG00000143633,FSAF1,DKFZp547B1713,C1orf131
6570,chr1,231520729,231528618,ENSG00000233461-gene,.,-,ENSG00000233461,ENSG00000233461,ENSG00000233461--ENSG00000233461,ENSG00000233461,TSNAX-DT,RP11-295G20.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65528,chr18,22609341,22933845,ENSG00000266850-gene,.,-,ENSG00000266850,ENSG00000266850,ENSG00000266850--ENSG00000266850,ENSG00000266850,RBBP8-AS1,AC090912.1,NaN
68211,chr19,28606688,28615229,ENSG00000266976-gene,.,+,ENSG00000266976,ENSG00000266976,ENSG00000266976--ENSG00000266976,ENSG00000266976,ERVE-5,AC079466.1,NaN
69598,chr19,52058490,52063703,ENSG00000260160-gene,.,-,ENSG00000260160,ENSG00000260160,ENSG00000260160--ENSG00000260160,ENSG00000260160,PROK2-DT,TCONS_00006091,NaN
71763,chr20,53453058,53504314,ENSG00000259723-gene,.,-,ENSG00000259723,ENSG00000259723,ENSG00000259723--ENSG00000259723,ENSG00000259723,TSHZ2-AS1,NaN,NaN
